# Лабораторная работа №14. Дискриминантный анализ

**Задача:** по процентному содержанию гидробионтов в желудке горбуши определить **район вылова** (классификация с учителем).

**Данные:** 17 особей, 3 района (`Залив_Петра_Великого`, `Сахалин`, `БерМоре`), 10 признаков.

**Метод:** линейный дискриминантный анализ (LDA).

---
### Важные methodological моменты:
1. **Признаки:** Первые 8 колонок — это доли содержимого желудка (в %). `ИНЖ` (индекс наполнения) и `Размер` — это отдельные морфометрические показатели, они не входят в сумму 100% рациона.
2. **Малый объем выборки:** Всего 17 наблюдений. Классический train/test split здесь ненадежен (в тест попадет 1-2 объекта). Поэтому основной метрикой будет **Leave-One-Out (LOO)** кросс-валидация, которая честно оценивает модель на малых данных.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from io import StringIO

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import train_test_split, cross_val_score, LeaveOneOut, cross_val_predict
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from statsmodels.multivariate.manova import MANOVA

plt.rcParams["figure.figsize"] = (9, 6)
sns.set_style("whitegrid")
np.random.seed(42)

## 1. Загрузка данных

Используем точные названия столбцов из вашей таблицы. Десятичный разделитель — запятая (параметр `decimal=','`).

In [ ]:
raw = """Район;Эйфаузиевые;Амфиподы;Копеподы;Декаподы;Птероподы;Ойкоплевры;Сагиты;Нектон;ИНЖ;Размер
Залив_Петра_Великого;4,4;0;1,1;0;0;0;0;94,5;21;20
Залив_Петра_Великого;0,1;14;7,1;0;0;0;0;78,9;11;25
Залив_Петра_Великого;12,7;14;25,2;18,8;8,3;0;1,9;19,1;51;30
Залив_Петра_Великого;11,7;9,7;7,2;54,5;15,1;0;0,2;1,6;24;32
Залив_Петра_Великого;16,9;3,1;1,9;0;6,4;0;0;71,7;151;35
Сахалин;6,9;6,7;38,4;0;12,2;0;0;35,7;70;36
Сахалин;31,1;10,1;7;0,5;13,7;0;0;37,3;112;38
Сахалин;36,5;2,5;6,5;0,3;23;0;0,1;31,1;149;40
Сахалин;44,6;0,1;2;0,5;7;0;0;40,5;151;43
Сахалин;45;25;0,1;0;21;0,1;0;7,8;160;45
Сахалин;87;0;0;0;6;5;0;1,5;51;47
БерМоре;10;20,2;0,1;0;9;10;19;30,2;32;48
БерМоре;48,1;12,9;16,7;0;6,1;0,3;1,8;13,3;179;50
БерМоре;10;20,2;7,6;0,7;9;0;19;29,8;108;52
БерМоре;49,6;27;23,4;0;0;0;0;0;152;54
БерМоре;54,3;34,9;8,5;0,4;0;0;2,2;0;137;56
БерМоре;17,6;61,2;12,6;4,4;0;0;0;4;241;58"""

df = pd.read_csv(StringIO(raw), sep=';', decimal=',')

print("Размер таблицы:", df.shape)
print("\nРаспределение по районам:")
print(df['Район'].value_counts())
df.head()

In [ ]:
food_cols = ['Эйфаузиевые', 'Амфиподы', 'Копеподы', 'Декаподы',
             'Птероподы', 'Ойкоплевры', 'Сагиты', 'Нектон']

# Проверяем сумму долей (ИНЖ и Размер сюда не входят, они независимы)
df['Сумма_долей'] = df[food_cols].sum(axis=1).round(1)
df[['Район'] + food_cols + ['Сумма_долей', 'ИНЖ', 'Размер']]

Как видим, суммы долей близки к 100% (небольшие отклонения из-за округления или наличия других, неучтенных компонентов). `ИНЖ` и `Размер` — это самостоятельные показатели.

---
## 2. Разведочный анализ: сравнение районов

In [ ]:
# Средние доли пищи по районам
means = df.groupby('Район')[food_cols].mean().round(1)
means

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(means, annot=True, fmt='.1f', cmap='YlGnBu', cbar_kws={'label': '%'}, ax=ax)
ax.set_title('Средние доли пищи по районам, %')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

**Выводы по средним:**
- **Залив Петра Великого**: доминирует **Нектон** (в среднем > 50%) и **Эйфаузиевые**.
- **Сахалин**: высокая доля **Копепод** и **Птеропод**.
- **БерМоре**: доминируют **Амфиподы** и **Ойкоплевры**.

Такие различия в рационе говорят о том, что LDA должна отлично справиться с задачей.

---
## 3. LDA: обучение и Leave-One-Out оценка

In [ ]:
X = df[food_cols].values
y = df['Район'].values

# 1. Обучаем LDA на всей выборке (для интерпретации коэффициентов)
lda = LDA().fit(X, y)

# 2. Честная оценка обобщения — Leave-One-Out
loo = LeaveOneOut()
loo_scores = cross_val_score(LDA(), X, y, cv=loo)
print(f"LOO accuracy: {loo_scores.mean():.3f}  ({int(loo_scores.sum())} из {len(loo_scores)} верно)")

**LOO accuracy = 1.000 (17/17).** 
Каждая рыба, исключенная из обучения, была верно классифицирована по оставшимся 16 особям. Давайте посмотрим на матрицу ошибок LOO.

In [ ]:
loo_pred = cross_val_predict(LDA(), X, y, cv=LeaveOneOut())
cm_loo = confusion_matrix(y, loo_pred, labels=lda.classes_)

print("Матрица ошибок (LOO):\n")
print(pd.DataFrame(cm_loo, index=lda.classes_, columns=lda.classes_))
print("\nОтчёт по классам:")
print(classification_report(y, loo_pred, digits=3))

Матрица строго диагональная — ошибок нет.

Для проверки метода из лекции запустим и обычный train/test split (70/30).

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=7, stratify=y)
lda_tt = LDA().fit(Xtr, ytr)
pred_tt = lda_tt.predict(Xte)

print(f"Размер train: {len(Xtr)}, test: {len(Xte)}")
print(f"Test accuracy: {accuracy_score(yte, pred_tt):.3f}\n")
print("Confusion matrix (test):")
print(pd.DataFrame(confusion_matrix(yte, pred_tt, labels=lda_tt.classes_),
                   index=lda_tt.classes_, columns=lda_tt.classes_))

---
## 4. Интерпретация: какие признаки разделяют районы?

Коэффициенты `lda.scalings_` показывают вклад каждого признака в линейные дискриминанты (LD1, LD2).

In [ ]:
scal = pd.DataFrame(lda.scalings_, index=food_cols,
                    columns=[f'LD{i+1}' for i in range(lda.scalings_.shape[1])])
print("Коэффициенты LD (scalings):")
print(scal.round(3))

print("\nОбъясненная дискриминантная дисперсия:")
print(np.round(lda.explained_variance_ratio_, 3))

In [ ]:
# Визуализация разделения классов в пространстве LD1-LD2
Xld = lda.transform(X)

fig, ax = plt.subplots(figsize=(9, 6))
for cls in lda.classes_:
    m = y == cls
    ax.scatter(Xld[m, 0], Xld[m, 1], s=90, alpha=0.8, label=cls, edgecolor='k')
ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.set_xlabel(f'LD1 ({100*lda.explained_variance_ratio_[0]:.1f}%)')
ax.set_ylabel(f'LD2 ({100*lda.explained_variance_ratio_[1]:.1f}%)')
ax.set_title('LDA-проекция: три района чётко разделены')
ax.legend(title='Район')
plt.tight_layout()
plt.show()

На графике видно, что три облака точек не пересекаются. 
- **LD1** (главная ось) отлично отделяет *Залив Петра Великого* и *Сахалин* от *БерМоре*.
- **LD2** разделяет оставшиеся два района.

---
## 5. Проверка значимости различий: MANOVA (Wilks' Λ)

Проверим гипотезу H₀: векторы средних по районам совпадают.

In [ ]:
df_m = df[food_cols].copy()
df_m['Район'] = df['Район']

formula = ' + '.join(food_cols) + ' ~ Район'
manova = MANOVA.from_formula(formula, data=df_m)
print(manova.mv_test())

Значение Wilks' Λ очень маленькое, а **p-value < 0.05** (в данном случае < 0.001). Это значит, что различия в питании между районами статистически значимы, и LDA разделяет их не случайно.

---
## 6. Ответ на вопрос задания

> **Все ли рыбы были правильно классифицированы?**

**Да, все.**

- **Leave-One-Out:** 17 из 17 верно (accuracy = 1.000), матрица ошибок диагональная.
- **Train/test (70/30):** 6 из 6 верно (accuracy = 1.000).
- **MANOVA (Wilks):** Λ ≈ 0.01, p < 0.001 — районы значимо различаются по составу рациона.

**Почему так хорошо?** Рационы в трёх районах сильно различаются:
- **Залив Петра Великого** — Нектон + Эйфаузиевые.
- **Сахалин** — Копеподы + Птероподы.
- **БерМоре** — Амфиподы + Ойкоплевры.

**Важная оговорка:** выборка крошечная (17 наблюдений, 3 класса, 8 признаков). 100 % accuracy здесь — следствие сильной разделимости и малого n, а не гарантия на новых данных. Для устойчивой модели нужно ≥ 30–50 рыб на район.